# Poisson Regression (Doctor Visits)

This example is part of the *Universal Inference* section, which explores running inference in a broad class of models by approximating what has no closed form. It fits a Poisson regression to a real data set: the doctor visits of 5190 adults in the 1977–1978 Australian Health Survey, the Poisson model of Cameron and Trivedi, *Regression Analysis of Count Data* (1998, Table 3.3). The example comes from the documentation of NaturalGradientMessagePassingRules.jl by Mykola Lukashchuk, which writes the model with a dedicated node for a Poisson count with a log-rate. Here the rate is `exp` of the linear predictor, a plain Julia function, and the model goes to `fallback = NaturalGradientPropagation()`, natural-gradient message passing (Lukashchuk et al., 2026), with no rule written for it. We check the posterior against self-normalised importance sampling from the joint Laplace approximation.

In [ ]:
using RxInfer, BayesBase, Distributions, LinearAlgebra, StableRNGs
using Downloads, Plots, PrettyTables, Printf
import ExpectationApproximations as EA

## Data

The data come with the R package AER (`data("DoctorVisits")`; source: the Journal of Applied Econometrics data archive, Mullahy 1997). We download them from the [Rdatasets](https://github.com/vincentarelbundock/Rdatasets) mirror the first time the example runs. One row per adult: `visits`, the number of visits to a doctor in the past two weeks; `gender`; `age` in years divided by 100; `income` in tens of thousands of dollars; `illness`, the number of illnesses in the past two weeks; `reduced`, the days of reduced activity in the past two weeks; `health`, a general health questionnaire score; and the yes/no indicators `private` (private insurance), `freepoor` (free insurance due to low income), `freerepat` (free insurance due to age or disability), `nchronic` and `lchronic` (a chronic condition not limiting and limiting activity). The factors become indicators, and age enters with its square:

In [ ]:
data_url = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/AER/DoctorVisits.csv"
data_file = joinpath("data", "DoctorVisits.csv")
if !isfile(data_file)
    mkpath(dirname(data_file))
    Downloads.download(data_url, data_file)
end

lines = readlines(data_file)
header = split(first(lines), ",")
column(name) = findfirst(==(name), header)
records = [split(line, ",") for line in lines[2:end]]
number(row, name) = parse(Float64, row[column(name)])
indicator(row, name, level) = row[column(name)] == level ? 1.0 : 0.0

names = ["intercept", "female", "age", "age²", "income", "illness", "reduced", "health", "private", "freepoor", "freerepat", "nchronic", "lchronic"]
X = map(records) do row
    age = number(row, "age")
    [
        1.0, indicator(row, "gender", "female"), age, age^2, number(row, "income"), number(row, "illness"),
        number(row, "reduced"), number(row, "health"), indicator(row, "private", "yes"), indicator(row, "freepoor", "yes"),
        indicator(row, "freerepat", "yes"), indicator(row, "nchronic", "yes"), indicator(row, "lchronic", "yes"),
    ]
end
y = [round(Int, number(row, "visits")) for row in records]
N, d = length(y), length(names)
@assert N == 5190 && d == 13 && extrema(x -> x[3], X) == (0.19, 0.72) # hide
(adults = N, coefficients = d, zero_counts = round(count(iszero, y) / N; digits = 3), largest_count = maximum(y))

## The model

$$\beta \sim \mathcal{N}(0, I), \qquad y_i \sim \mathrm{Poisson}\big(\exp(x_i^\top \beta)\big),$$

with $x_i$ the intercept and the twelve covariates of adult $i$: 13 coefficients. The linear predictor is RxInfer's `dot` node, and the rate a deterministic node of `exp`:

In [ ]:
@model function doctor_visits(y, X, d)
    β ~ MvNormalMeanCovariance(zeros(d), diageye(d))
    for i in eachindex(y)
        z[i] := dot(X[i], β)
        λ[i] := exp(z[i])
        y[i] ~ Poisson(λ[i])
    end
end

`exp` has no rules. Under `fallback = NaturalGradientPropagation()`:

- the `Poisson` node sends its own message towards the rate, $\Gamma(\lambda_i \mid y_i + 1, 1)$, and the fallback pulls it back through `exp` towards $z_i$: $\log m(z_i) = y_i z_i - e^{z_i}$, the Poisson log-likelihood of the log-rate, exactly;
- this message is not a Gaussian, so $z_i$ keeps a Gaussian site for it, which moves at each iteration by a damped step towards its target, the natural gradient of the expected log-likelihood under $z_i$'s marginal;
- `dot` passes the sites to $\beta$ by its exact rules, and $\beta$'s posterior is the prior times 5190 Gaussian messages, exact.

At the fixed point, each site of $z_i$ is the Gaussian of conjugate-computation variational inference, and with one latent vector, $\beta$, the posterior is the variational Gaussian of the joint, the one that minimises $\mathrm{KL}(q \,\|\, p(\beta \mid y))$.

Each $z_i$ has two edges, towards `dot` and towards `exp`, and the site of the second needs the first's message, which needs $\beta$'s, which needs every other site: a loop through $\beta$, which an initial message starts. As in the original example, we start each $z_i$ at $\log(y_i + 1)$, so that the first step is a weighted regression of the log-counts. The fixed point does not depend on it.

We keep the posteriors of the $z_i$, which give the predicted rates below. The free energy would need the marginal of every rate $\lambda_i$, and a Gamma site for each, which costs much more than the inference itself; we leave it out.

In [ ]:
seconds = @elapsed result = infer(
    model = doctor_visits(X = X, d = d),
    data = (y = y,),
    initialization = @initialization(begin
        μ(z) = NormalMeanVariance.(log.(y .+ 1), 1.0)
    end),
    iterations = 30,
    fallback = NaturalGradientPropagation(),
    returnvars = (β = KeepLast(), z = KeepLast()),
    report = true,
)
q_β = result.posteriors[:β]
@printf("%.1f seconds, compilation included\n", seconds)

The report shows the 5190 sites of $z$, by the fallback's messages through `exp`, and that every site had settled at the last of the 30 iterations. $\beta$ and the rates are exact: $\beta$'s posterior is a product of Gaussian messages, and no one reads the rates' marginals.

In [ ]:
@assert isempty(RxInfer.diagnostics(result.report)) # hide
@assert all(i -> result.report.approximations.variables[(:z, i)].sites.outcome.method === :natural_gradient, (1, 2500, N)) # hide
@assert !RxInfer.is_approximated(result.report.approximations.variables[(:β, nothing)]) # hide
result.report.approximations

## The reference: importance sampling

The log-posterior of $\beta$ is concave, and the joint Laplace approximation, a Gaussian at its mode with the inverse Hessian as covariance, is close to it. Self-normalised importance sampling corrects it: draws from the Laplace Gaussian, weighted by the posterior over the proposal. ExpectationApproximations.jl has both, `laplace` and the `SNIS` backend, which takes a target with a log-density.

In [ ]:
struct Posterior <: ContinuousMultivariateDistribution end
Base.length(::Posterior) = d
Distributions.logpdf(::Posterior, β::AbstractVector) = -sum(abs2, β) / 2 + sum(y[i] * dot(X[i], β) - exp(dot(X[i], β)) for i in eachindex(y))

laplace = EA.laplace(β -> logpdf(Posterior(), β), zeros(d))
proposal = MvNormal(laplace.mode, Symmetric(cov(laplace.distribution)))
draws = BayesBase.prepare_expectation(EA.SNIS(proposal, 20_000, StableRNG(1)), Posterior())
m_ref = BayesBase.expectation(draws, identity)
Σ_ref = BayesBase.expectation(draws, β -> (β - m_ref) * (β - m_ref)')
s_ref = sqrt.(diag(Σ_ref))
ess = EA.effective_sample_size(draws)
@printf("Laplace converged: %s; effective sample size %.0f of 20000\n", laplace.converged, ess)
@assert laplace.converged && ess > 15_000 # hide

The weights are nearly uniform: the effective sample size is above 15 000 of the 20 000 draws, so the Laplace Gaussian is close to the posterior, and the Monte Carlo standard error of each posterior mean is below 1% of its posterior standard deviation.

## The coefficients

In [ ]:
m, s = mean(q_β), sqrt.(diag(cov(q_β)))
gap(a) = abs.(a .- m_ref) ./ s_ref
rows = hcat(names, m, s, m_ref, s_ref, gap(m), gap(laplace.mode))
@assert maximum(gap(m)) < 0.02 && maximum(abs, s ./ s_ref .- 1) < 0.015 # hide
@assert maximum(gap(laplace.mode)) > 3 * maximum(gap(m)) && 0.04 < maximum(gap(laplace.mode)) < 0.07 # hide
pretty_table(rows; column_labels = ["coefficient", "NGMP mean", "NGMP sd", "SNIS mean", "SNIS sd", "|NGMP - SNIS| / sd", "|Laplace - SNIS| / sd"], formatters = [fmt__printf("%.4f", 2:5), fmt__printf("%.3f", 6:7)], fit_table_in_display_horizontally = false)

Natural-gradient propagation agrees with importance sampling: every posterior mean is within 0.02 posterior standard deviations of the sampled one, about the Monte Carlo error, and every standard deviation within 1.5%. Laplace's mean, the mode, is about 0.05 standard deviations away at most, more than three times further: with 5190 counts the posterior is nearly Gaussian, and the variational Gaussian follows what little skew is left.

In [ ]:
plt = scatter(m, 1:d; xerror = 2s, yticks = (1:d, names), yflip = true, color = :blue, markersize = 4, label = "posterior mean ± 2 sd", xlabel = "coefficient (log-rate scale)", legend = :topright, size = (720, 440))
vline!(plt, [0.0]; color = :black, alpha = 0.3, label = "")

Illnesses, days of reduced activity and the health score raise the visit rate, and women visit more often; income and free insurance for the poor lower it, each by more than two posterior standard deviations from zero. Age and its square are nearly collinear on this sample, with ages from 19 to 72, so age in hundreds lies in $[0.19, 0.72]$: the data pin down only a combination of the two, and their coefficients are strongly anti-correlated, with wide posteriors on each.

In [ ]:
lower, upper = m .- 2s, m .+ 2s
coefficient(name) = findfirst(==(name), names)
@assert all(lower[coefficient(n)] > 0 for n in ("illness", "reduced", "health", "female")) # hide
@assert all(upper[coefficient(n)] < 0 for n in ("income", "freepoor")) # hide
correlation = cov(q_β)[3, 4] / (s[3] * s[4])
@printf("correlation of the age and age² coefficients: %.3f\n", correlation)
@assert correlation < -0.95 # hide

## A posterior predictive check

The observed distribution of the counts against the predicted one. Each adult's rate is $e^{z_i}$, and its posterior mean the mean of a log-normal, $\exp(m_i + v_i / 2)$, with $m_i$ and $v_i$ the mean and variance of $z_i$'s posterior. They are those of $x_i^\top \beta$ under $\beta$'s posterior, to the precision of the iterations. The predicted frequency of each count is the average of the Poisson probabilities over the adults.

In [ ]:
q_z = result.posteriors[:z]
@assert maximum(abs, mean.(q_z) .- [dot(x, m) for x in X]) < 1.0e-6 && maximum(abs, var.(q_z) .- [dot(x, cov(q_β), x) for x in X]) < 1.0e-6 # hide
rates = [exp(mean(q) + var(q) / 2) for q in q_z]
predicted = [sum(pdf(Poisson(λ), k) for λ in rates) / N for k in 0:maximum(y)]
observed = [count(==(k), y) / N for k in 0:maximum(y)]
tail_observed, tail_predicted = 1 - sum(observed[1:3]), 1 - sum(predicted[1:3])
@assert observed[1] > predicted[1] && tail_observed > tail_predicted && observed[2] < predicted[2] # hide
pretty_table(hcat(0:4, observed[1:5], predicted[1:5]); column_labels = ["visits", "observed frequency", "predicted frequency"], formatters = [fmt__printf("%.4f", 2:3)], fit_table_in_display_horizontally = false)
@printf("three visits or more: observed %.4f, predicted %.4f\n", tail_observed, tail_predicted)

The model follows the shape of the data, but predicts too few adults with no visit, too many with one, and too few with three or more: the counts are overdispersed relative to a Poisson, as Cameron and Trivedi discuss, which a negative binomial likelihood would address. That is a property of the Poisson model, not of the inference.

## Summary

- The model is written as it reads, with `exp` a plain function between the linear predictor and RxInfer's `Poisson` node, and `fallback = NaturalGradientPropagation()` fits it without a rule for `exp`: one Gaussian site per log-rate, 30 iterations.
- The posterior agrees with importance sampling to within 0.02 standard deviations in every mean and 1.5% in every spread; Laplace's mode is further off.

## References

- A. C. Cameron and P. K. Trivedi (1998). *Regression Analysis of Count Data*. Cambridge University Press.
- J. Mullahy (1997). Heterogeneity, excess zeros, and the structure of count data models. *Journal of Applied Econometrics* 12(3).
- M. Lukashchuk, K. Yemets, A. Ledbetter and İ. Şenöz (2026). Information geometry of message passing. arXiv:2608.15922.